<a href="https://colab.research.google.com/github/SasmitDey/fake_news_verifier/blob/main/fake_news_verifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

__IMPORTING REQUIRED LIBRARIES__

_Required libraries (fix order):_
1. polars (replacing pandas (novelty))
2. numpy
3. gensim
4. sentence-transformers
5. matplotlib
6. scikit-learn
7. os
8. dotenv
9. kagglehub
10. torch (for cuda)

In [4]:
!pip install -qU polars sentence-transformers scikit-learn dotenv kagglehub

In [5]:
!nvidia-smi

Fri May  8 04:44:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os
from dotenv import load_dotenv

import polars as pl
import numpy
# import gensim
import torch

In [7]:
#wv = gensim.models.KeyedVectors.load_word2vec_format('word2vec_format/GoogleNews-vectors-negative300.bin', binary=True)

__LOADING DATASET__

_Dataset Details:_
1. Name - ISOT Fake News Dataset
2. Link - https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets/data
3. Number of rows - 20,826 (True) and 17,903 (False)

In [8]:
# load_dotenv()

# kaggle_key = os.getenv('KAGGLE_API_TOKEN')
# kaggle_username = os.getenv('KAGGLE_USERNAME')

In [9]:
import kagglehub
# from kagglehub import KaggleDatasetAdapter

In [10]:
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")

Using Colab cache for faster access to the 'fake-news-detection-datasets' dataset.


In [11]:
# true_df = pl.read_csv('dataset/News _dataset/True.csv')
# fake_df = pl.read_csv('dataset/News _dataset/Fake.csv')

In [12]:
true_df = pl.read_csv('/content/True.csv')
fake_df = pl.read_csv('/content/Fake.csv')

__PROCESSING DATASET__

_We drop the date column from both datasets_
_We add a new column called label for both datasets, which adds a 1 label for true and -1 label for false (this will help in clssification later)_

___VERY IMPORTANT:___
_Using regular expression, we delete the mention of Reuters from true.csv as that will lead to model overfitting_

_We then merge both datasets to be later split for training and testing our RandomForest model (using cuML if possible)_

In [13]:
true_df.head()

title,text,subject,date
str,str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews""","""December 31, 2017 """
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews""","""December 29, 2017 """
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews""","""December 31, 2017 """
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews""","""December 30, 2017 """
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews""","""December 29, 2017 """


In [14]:
true_df = true_df.drop('date')

In [15]:
true_df.head()

title,text,subject
str,str,str
"""As U.S. budget fight looms, Re…","""WASHINGTON (Reuters) - The hea…","""politicsNews"""
"""U.S. military to accept transg…","""WASHINGTON (Reuters) - Transge…","""politicsNews"""
"""Senior U.S. Republican senator…","""WASHINGTON (Reuters) - The spe…","""politicsNews"""
"""FBI Russia probe helped by Aus…","""WASHINGTON (Reuters) - Trump c…","""politicsNews"""
"""Trump wants Postal Service to …","""SEATTLE/WASHINGTON (Reuters) -…","""politicsNews"""


In [16]:
fake_df.head()

title,text,subject,date
str,str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News""","""December 31, 2017"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News""","""December 31, 2017"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News""","""December 30, 2017"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News""","""December 29, 2017"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News""","""December 25, 2017"""


In [17]:
fake_df = fake_df.drop("date")
fake_df.head()

title,text,subject
str,str,str
""" Donald Trump Sends Out Embarr…","""Donald Trump just couldn t wis…","""News"""
""" Drunk Bragging Trump Staffer …","""House Intelligence Committee C…","""News"""
""" Sheriff David Clarke Becomes …","""On Friday, it was revealed tha…","""News"""
""" Trump Is So Obsessed He Even …","""On Christmas day, Donald Trump…","""News"""
""" Pope Francis Just Called Out …","""Pope Francis used his annual C…","""News"""


In [18]:
true_df = true_df.with_columns(pl.lit(1).alias("label"))
fake_df = fake_df.with_columns(pl.lit(0).alias("label"))

# 3. Remove "Reuters" mentions (VERY IMPORTANT for leakage)
true_df = true_df.with_columns(
    pl.col("text").str.replace_all(r"(?i)reuters", "")
)

# Optional: also clean titles if needed
true_df = true_df.with_columns(
    pl.col("text").str.replace(
        r"^[A-Z\s\.,-]+\s*\(Reuters\)\s*-\s*",
        ""
    )
)
# 4. Merge datasets
df = pl.concat([true_df, fake_df])

# 5. Shuffle dataset (important before split)
df = df.sample(fraction=1.0, shuffle=True)

# 6. Train-test split
train_size = int(0.8 * df.height)

In [19]:
df.head()

title,text,subject,label
str,str,str,i32
""" Hyperventilating Christians F…","""The Satanic Temple just forced…","""News""",0
""" One Of Bernie Sanders’ Bigges…","""Ta-Nehisi Coates sat down with…","""News""",0
"""FIRST LADY Makes Surprise Visi…",""" ""","""politics""",0
"""BREITBART’S JOEL POLLACK Brill…","""Watch Joy Behar as she hilario…","""Government News""",0
"""Pence: Trump will focus fast o…","""() - The Trump administration …","""politicsNews""",1


_We will use cuda if available to speed up SBERT embedding process_

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


__VECTORISING TEXT CONTENT__

In [21]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124


In [22]:
!pip install -qU sentence-transformers

In [23]:

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Torch Version: {torch.__version__}")

CUDA Available: True
Torch Version: 2.6.0+cu124


In [24]:
# Install PyTorch 2.6.0 and compatible ecosystem libraries
!pip install -qU torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
# !pip install -qU torchcodec==0.3.0

import torch
print(f"Current Torch Version: {torch.__version__}")

Current Torch Version: 2.6.0+cu124


In [25]:
# Install torchcodec version compatible with PyTorch 2.1
!pip install -qU torchcodec==0.2.1

In [26]:
print('Installing FFmpeg...')
!apt-get update && apt-get install -y ffmpeg
print('FFmpeg installed successfully.')

Installing FFmpeg...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry miss

In [27]:
# Install necessary system libraries for multimedia codecs
print('Installing libsm6 and libxext6...')
!apt-get update && apt-get install -y libsm6 libxext6
print('libsm6 and libxext6 installed successfully.')

Installing libsm6 and libxext6...
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.li

In [28]:
!pip -q install "numpy<2.0.0"

from sentence_transformers import SentenceTransformer

In [29]:
# model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
# model = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", device=device, trust_remote_code=True)

model = SentenceTransformer(
    "BAAI/bge-m3",
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [30]:
model.half()

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [31]:
# model.tokenizer.model_max_length = 8192
# model.tokenizer.padding_side = "right"
targets = ["title", "text", "subject"]

for column in targets:
    print(f"Encoding {column}...")
    sentences = (
        df.select(pl.col(column))
        .cast(pl.String)
        .fill_null("")
        .to_series()
        .to_list()
    )
    embeddings = model.encode(
        sentences,
        show_progress_bar=True,
        convert_to_numpy=True,
        batch_size=32,
        normalize_embeddings=True
    )
    df = df.with_columns([
        pl.Series(column, embeddings.tolist())
    ])

Encoding title...


Batches:   0%|          | 0/1404 [00:00<?, ?it/s]

Encoding text...


Batches:   0%|          | 0/1404 [00:00<?, ?it/s]

Encoding subject...


Batches:   0%|          | 0/1404 [00:00<?, ?it/s]

In [32]:
df.head()

title,text,subject,label
list[f64],list[f64],list[f64],i32
"[0.027084, -0.00576, … 0.03096]","[0.043549, 0.030487, … 0.01973]","[-0.016266, 0.036011, … -0.004494]",0
"[-0.040558, -0.008347, … -0.006413]","[-0.03949, -0.00671, … -0.015205]","[-0.016266, 0.036011, … -0.004494]",0
"[0.005455, -0.014336, … -0.018417]","[-0.029861, -0.00639, … 0.001993]","[0.005348, 0.035797, … -0.029724]",0
"[-0.042114, 0.008125, … -0.0103]","[-0.021591, -0.012535, … 0.026764]","[-0.039948, 0.025497, … -0.056671]",0
"[-0.029877, 0.051483, … -0.002762]","[-0.020187, 0.023727, … -0.080688]","[-0.003187, 0.03476, … -0.037994]",1


In [35]:
df.write_parquet(
    "df_embedded.parquet",
    compression="zstd"
)

__TRAINING MODEL ON DATASET__

_We use RandomForest from cuML (or scikit-learn for fallback) to train our model to predict news classifications_

In [34]:
# try:
#     print("Importing cuML model")
#     from cuml.ensemble import RandomForestClassifier
# except Exception:
#     print("Importing scikit-learn model")
#     from sklearn.ensemble import RandomForestClassifier